# 08 — Kepler Final Held-Out Test Evaluation

This notebook evaluates **three previously selected models** on the same untouched test set: the tuned Decision Tree, optimized 64–32 MLP, and tuned XGBoost. Every model is fitted on the **full training split only**, and the held-out test split is used **once for final reporting**. There is **no hyperparameter search, early stopping, or threshold selection on test data**.

**Run all cells from top to bottom** in VS Code after placing this notebook beside `Kepler_Datasets/` or `Kepler_Datasets(1).zip`. Install dependencies if needed with `python -m pip install pandas numpy scikit-learn xgboost matplotlib`.

The code uses the selected model configurations from Notebooks 02, 04, and 06. The 0.50 probability threshold remains unchanged for all models.

## 1. Imports and data loading

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import io
import time
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay
)
from sklearn.exceptions import ConvergenceWarning
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)

DATA_DIR = Path("Kepler_Datasets")
ZIP_CANDIDATES = [Path("Kepler_Datasets(1).zip"), Path("Kepler_Datasets.zip")]


def load_csv(name):
    candidates = [DATA_DIR / name, Path(name)]
    for path in candidates:
        if path.is_file():
            return pd.read_csv(path)
    for archive in ZIP_CANDIDATES:
        if archive.is_file():
            with zipfile.ZipFile(archive) as z:
                if name in z.namelist():
                    return pd.read_csv(io.BytesIO(z.read(name)))
    raise FileNotFoundError(f"Cannot find {name} in Kepler_Datasets/, current folder, or dataset ZIP")

X_train = load_csv("X_train.csv")
X_test = load_csv("X_test.csv")
y_train = load_csv("y_train.csv")["target"].astype(int).to_numpy()
y_test = load_csv("y_test.csv")["target"].astype(int).to_numpy()
train_ids = load_csv("train_ids.csv")
test_ids = load_csv("test_ids.csv")

assert len(X_train) == len(y_train) == len(train_ids)
assert len(X_test) == len(y_test) == len(test_ids)
assert list(X_train.columns) == list(X_test.columns), "Train/test feature columns differ"
assert X_train.shape[1] == 19, "Expected the same 19 numerical features used for tuning"
assert set(np.unique(y_train)) == {0, 1} and set(np.unique(y_test)) == {0, 1}
assert "kepid" in train_ids.columns and "kepid" in test_ids.columns
assert set(train_ids["kepid"]).isdisjoint(set(test_ids["kepid"])), "Host-star leakage: overlapping kepid values"

print(f"Training observations: {len(X_train):,}")
print(f"Held-out test observations: {len(X_test):,}")
print(f"Features: {X_train.shape[1]}")
print(f"Train class counts: {dict(pd.Series(y_train).value_counts().sort_index())}")
print(f"Test class counts: {dict(pd.Series(y_test).value_counts().sort_index())}")
print("Train/test host-star overlap: 0")

## 2. Locked model configurations

In [ ]:
models = {
    "Tuned Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("tree", DecisionTreeClassifier(
            criterion="gini", max_depth=8, min_samples_split=10,
            min_samples_leaf=10, class_weight="balanced", random_state=42
        ))
    ]),
    "Optimized MLP (64-32)": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
            alpha=0.01, learning_rate_init=0.001, batch_size=128,
            max_iter=90, n_iter_no_change=20, tol=1e-4,
            early_stopping=False, random_state=42
        ))
    ]),
    "Tuned XGBoost": XGBClassifier(
        n_estimators=600, max_depth=8, learning_rate=0.05,
        reg_lambda=1, reg_alpha=0, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=1.0,
        objective="binary:logistic", eval_metric="logloss",
        tree_method="hist", max_bin=64, random_state=42, n_jobs=1
    )
}

print("Locked configurations:")
for name in models:
    print(" -", name)

## 3. Fit on full training set; evaluate once on held-out test set

In [ ]:
scores = []
prediction_rows = []
confusion_matrices = {}
roc_models = {}

for name, model in models.items():
    started = time.perf_counter()
    model.fit(X_train, y_train)
    training_seconds = time.perf_counter() - started

    started = time.perf_counter()
    predicted = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    prediction_seconds = time.perf_counter() - started

    tn, fp, fn, tp = confusion_matrix(y_test, predicted, labels=[0, 1]).ravel()
    scores.append({
        "model": name,
        "test_F1": f1_score(y_test, predicted),
        "test_precision": precision_score(y_test, predicted, zero_division=0),
        "test_recall": recall_score(y_test, predicted, zero_division=0),
        "test_accuracy": accuracy_score(y_test, predicted),
        "test_ROC_AUC": roc_auc_score(y_test, probabilities),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "test_FPR": fp / (fp + tn) if (fp + tn) else float("nan"),
        "test_FNR": fn / (fn + tp) if (fn + tp) else float("nan"),
        "fit_seconds": training_seconds,
        "predict_seconds": prediction_seconds
    })
    confusion_matrices[name] = confusion_matrix(y_test, predicted, labels=[0, 1])
    roc_models[name] = probabilities

    for i, (actual, pred, prob) in enumerate(zip(y_test, predicted, probabilities)):
        prediction_rows.append({
            "model": name, "test_row": i,
            "kepid": test_ids["kepid"].iloc[i],
            "kepoi_name": test_ids["kepoi_name"].iloc[i] if "kepoi_name" in test_ids else "",
            "actual": int(actual), "predicted": int(pred),
            "positive_probability": float(prob),
            "error_type": "FP" if actual == 0 and pred == 1 else "FN" if actual == 1 and pred == 0 else "correct"
        })
    print(f"{name}: test F1={scores[-1]['test_F1']:.4f}; ROC-AUC={scores[-1]['test_ROC_AUC']:.4f}")

results = pd.DataFrame(scores).sort_values("test_F1", ascending=False).reset_index(drop=True)
predictions = pd.DataFrame(prediction_rows)
results.to_csv("kepler_final_test_comparison.csv", index=False)
predictions.to_csv("kepler_final_test_predictions.csv", index=False)

print("\nFINAL HELD-OUT TEST RESULTS")
display(results.round(4))
print("Saved kepler_final_test_comparison.csv and kepler_final_test_predictions.csv")

## 4. Final test comparison charts

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
metric_cols = ["test_F1", "test_precision", "test_recall", "test_accuracy", "test_ROC_AUC"]
results.set_index("model")[metric_cols].plot(kind="bar", ax=ax, rot=10)
ax.set_ylim(0, 1.02)
ax.set_ylabel("Score")
ax.set_title("Final held-out test performance: three Kepler models")
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
fig.savefig("kepler_final_test_metrics.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
for name, probabilities in roc_models.items():
    RocCurveDisplay.from_predictions(y_test, probabilities, name=name, ax=ax)
ax.set_title("Final held-out test ROC curves")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig("kepler_final_test_roc.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Confusion matrices and false-positive comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, (name, cm) in zip(axes, confusion_matrices.items()):
    ConfusionMatrixDisplay(cm, display_labels=["False positive", "Candidate"]).plot(
        ax=ax, colorbar=False, values_format="d"
    )
    ax.set_title(name)
fig.tight_layout()
fig.savefig("kepler_final_test_confusion_matrices.png", dpi=180, bbox_inches="tight")
plt.show()

print("False positive comparison (lower is better, holding recall trade-offs in mind):")
display(results[["model", "FP", "FN", "test_FPR", "test_FNR", "test_F1"]].round(4))

## Reporting guidance

- These scores are **held-out test results**, not five-fold cross-validation averages. Report them separately from the validation scores in the earlier notebooks.
- The training–validation gap reported during model development is **not** the train–test gap; this notebook does not use test results to retune models.
- Choose the final model based on the previously defined validation criterion. Do not revise hyperparameters, thresholds, features, or preprocessing after inspecting the held-out test performance.
- Output files created in the current working directory: `kepler_final_test_comparison.csv`, `kepler_final_test_predictions.csv`, `kepler_final_test_metrics.png`, `kepler_final_test_roc.png`, and `kepler_final_test_confusion_matrices.png`.
